# 02 · Feature Engineering + Modelo ML
Forecast de ventas · Motor 1 · LightGBM

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','pandas','numpy','lightgbm','scikit-learn','plotly','python-dotenv','supabase','openpyxl'])
print('OK')

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, pickle, os, sys, warnings
from pathlib import Path
from datetime import date
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve()))
from conexion import get_supabase_client, supabase_to_df, df_to_supabase
sb = get_supabase_client()

## 1 · Cargar datos de Supabase

In [ ]:
print('Cargando ventas...')
df_v = supabase_to_df(sb, 'fact_ventas_diarias')
df_v['fecha'] = pd.to_datetime(df_v['fecha'])
df_tiendas = supabase_to_df(sb, 'dim_tiendas')
df_eventos = supabase_to_df(sb, 'dim_eventos')
df_eventos['fecha'] = pd.to_datetime(df_eventos['fecha'])
print(f'Ventas: {len(df_v):,} | Tiendas: {len(df_tiendas)} | Eventos: {len(df_eventos)}')

In [ ]:
# Agregar a nivel semanal
df_v['semana'] = df_v['fecha'].dt.to_period('W').dt.start_time
df = df_v.groupby(['semana','tienda_id','tipo_producto','familia']).agg(
    unidades=('unidades_vendidas','sum'),
    valor_venta=('valor_venta','sum'),
    margen_bruto=('margen_bruto','sum'),
    descuento_pct_avg=('descuento_pct','mean'),
    pct_precio_pleno=('es_precio_pleno','mean')
).reset_index()
df['tienda_id'] = df['tienda_id'].astype(int)
df_tiendas['tienda_id'] = df_tiendas['tienda_id'].astype(int)
df = df.merge(df_tiendas[['tienda_id','ciudad','formato','segmento_cliente','indice_rotacion','metros_cuadrados']],on='tienda_id',how='left')
print(f'{len(df):,} filas semanales | {df.semana.min().date()} -> {df.semana.max().date()}')

## 2 · Feature Engineering

In [ ]:
df['semana_iso'] = df['semana'].dt.isocalendar().week.astype(int)
df['mes']        = df['semana'].dt.month
df['anio']       = df['semana'].dt.year
df['trimestre']  = df['semana'].dt.quarter
df['es_quincena']= df['semana'].dt.day.isin([14,15,16,28,29,30]).astype(int)
df['semana_sin'] = np.sin(2*np.pi*df['semana_iso']/52)
df['semana_cos'] = np.cos(2*np.pi*df['semana_iso']/52)
df['mes_sin']    = np.sin(2*np.pi*df['mes']/12)
df['mes_cos']    = np.cos(2*np.pi*df['mes']/12)
df = df.sort_values(['tienda_id','tipo_producto','semana'])
grp = df.groupby(['tienda_id','tipo_producto'])['unidades']
df['lag_1sem']      = grp.shift(1)
df['lag_2sem']      = grp.shift(2)
df['lag_4sem']      = grp.shift(4)
df['lag_8sem']      = grp.shift(8)
df['lag_52sem']     = grp.shift(52)
df['rolling_4sem']  = grp.shift(1).rolling(4,min_periods=1).mean().reset_index(0,drop=True)
df['rolling_8sem']  = grp.shift(1).rolling(8,min_periods=1).mean().reset_index(0,drop=True)
df['rolling_12sem'] = grp.shift(1).rolling(12,min_periods=1).mean().reset_index(0,drop=True)
df['rolling_std4']  = grp.shift(1).rolling(4,min_periods=1).std().reset_index(0,drop=True)
df['tendencia']     = (df['rolling_4sem']/df['rolling_8sem'].replace(0,np.nan)).fillna(1)
print('Features de tiempo y demanda OK')

In [ ]:
def crear_features_eventos(df, df_eventos):
    df = df.copy()
    for _,ev in df_eventos.iterrows():
        nombre=ev['nombre_evento']; fecha_ev=pd.to_datetime(ev['fecha'])
        alcance=ev['alcance']; pre=int(ev['semanas_anticipacion']); post=int(ev['semanas_rebote'])
        for s in range(-pre,post+1):
            sem_obj=fecha_ev+pd.Timedelta(weeks=s)
            tag=f"ev_{nombre[:12]}"+(f"_pre{abs(s)}" if s<0 else "" if s==0 else f"_post{s}")
            if tag not in df.columns: df[tag]=0
            mask=df['semana']==sem_obj
            if alcance=='nacional': df.loc[mask,tag]=1
            else: df.loc[mask&(df['ciudad']==alcance),tag]=1
    return df
df = crear_features_eventos(df, df_eventos)
cols_eventos = [c for c in df.columns if c.startswith('ev_')]
print(f'{len(cols_eventos)} features de eventos')

In [ ]:
encoders={}
for col in ['tipo_producto','familia','ciudad','formato','segmento_cliente']:
    enc=LabelEncoder()
    df[f'{col}_enc']=enc.fit_transform(df[col].astype(str))
    encoders[col]=enc
print('Encoding OK')

## 3 · Entrenamiento LightGBM

In [ ]:
FEATURES=['semana_iso','mes','trimestre','anio','semana_sin','semana_cos','mes_sin','mes_cos','es_quincena',
    'lag_1sem','lag_2sem','lag_4sem','lag_8sem','lag_52sem','rolling_4sem','rolling_8sem','rolling_12sem','rolling_std4','tendencia',
    'descuento_pct_avg','pct_precio_pleno',
    'tipo_producto_enc','familia_enc','ciudad_enc','formato_enc','segmento_cliente_enc',
    'indice_rotacion','metros_cuadrados']+cols_eventos
fecha_corte=df['semana'].max()-pd.Timedelta(weeks=12)
df_train=df[df['semana']<=fecha_corte].dropna(subset=FEATURES)
df_test =df[df['semana']> fecha_corte].dropna(subset=FEATURES)
X_train,y_train=df_train[FEATURES],df_train['unidades']
X_test, y_test =df_test[FEATURES], df_test['unidades']
print(f'Train:{len(X_train):,} | Test:{len(X_test):,} | Features:{len(FEATURES)}')

In [ ]:
modelo=lgb.LGBMRegressor(objective='regression',metric='mae',n_estimators=500,learning_rate=0.05,
    num_leaves=63,min_child_samples=20,subsample=0.8,colsample_bytree=0.8,
    reg_alpha=0.1,reg_lambda=0.1,random_state=42,verbose=-1)
modelo.fit(X_train,y_train,eval_set=[(X_test,y_test)],
    callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(100)])
print('Modelo entrenado')

In [ ]:
y_pred=np.maximum(0,modelo.predict(X_test))
mape=mean_absolute_percentage_error(y_test[y_test>0],y_pred[y_test>0])*100
mae=mean_absolute_error(y_test,y_pred)
print(f'MAPE: {mape:.1f}% | MAE: {mae:.1f} unidades')
imp=pd.DataFrame({'feature':FEATURES,'importance':modelo.feature_importances_}).sort_values('importance',ascending=False).head(15)
px.bar(imp,x='importance',y='feature',orientation='h',title='Top 15 features',color='importance',
       color_continuous_scale='Blues').update_layout(height=420,yaxis={'categoryorder':'total ascending'}).show()

## 4 · Forecast 8 semanas → Supabase

In [ ]:
fecha_hoy=df['semana'].max()
combinaciones=df[['tienda_id','tipo_producto','familia','ciudad','formato','segmento_cliente','indice_rotacion','metros_cuadrados']].drop_duplicates()
ultimo=df.sort_values('semana').groupby(['tienda_id','tipo_producto']).last().reset_index()
rows=[]
for h in range(1,9):
    sem_obj=fecha_hoy+pd.Timedelta(weeks=h)
    df_f=combinaciones.copy()
    df_f['semana']=sem_obj
    df_f['semana_iso']=sem_obj.isocalendar()[1]
    df_f['mes']=sem_obj.month; df_f['anio']=sem_obj.year; df_f['trimestre']=(sem_obj.month-1)//3+1
    df_f['es_quincena']=int(sem_obj.day in list(range(14,18))+list(range(28,32)))
    df_f['semana_sin']=np.sin(2*np.pi*df_f['semana_iso']/52)
    df_f['semana_cos']=np.cos(2*np.pi*df_f['semana_iso']/52)
    df_f['mes_sin']=np.sin(2*np.pi*df_f['mes']/12)
    df_f['mes_cos']=np.cos(2*np.pi*df_f['mes']/12)
    df_f=df_f.merge(ultimo[['tienda_id','tipo_producto','rolling_4sem','rolling_8sem','rolling_12sem',
        'rolling_std4','tendencia','lag_1sem','lag_2sem','lag_4sem','lag_8sem','lag_52sem',
        'descuento_pct_avg','pct_precio_pleno']],on=['tienda_id','tipo_producto'],how='left')
    for col in ['tipo_producto','familia','ciudad','formato','segmento_cliente']:
        df_f[f'{col}_enc']=encoders[col].transform(df_f[col].astype(str))
    for col in cols_eventos: df_f[col]=0
    df_f=crear_features_eventos(df_f,df_eventos).fillna(0)
    preds=np.maximum(0,modelo.predict(df_f[FEATURES]))
    df_f['forecast_medio']=preds; df_f['forecast_bajo']=preds*0.80; df_f['forecast_alto']=preds*1.20
    df_f['fecha_ejecucion']=str(date.today()); df_f['semana_objetivo']=str(sem_obj.date()); df_f['error_mape']=round(mape,2)
    rows.append(df_f[['fecha_ejecucion','semana_objetivo','tienda_id','tipo_producto','familia','forecast_bajo','forecast_medio','forecast_alto','error_mape']])
df_forecast=pd.concat(rows,ignore_index=True)
df_to_supabase(sb, df_forecast, 'output_forecast_semanal', limpiar_hoy=True)
print(f'Forecast guardado: {len(df_forecast):,} filas')

In [ ]:
os.makedirs('../outputs',exist_ok=True)
with open('../outputs/modelo_lgbm.pkl','wb') as f:
    pickle.dump({'modelo':modelo,'features':FEATURES,'encoders':encoders,'mape':mape,'mae':mae},f)
print('Modelo guardado en outputs/modelo_lgbm.pkl')
print('Continuar con notebook 03')